In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import ast
import pickle

import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader

import sklearn as sk
from sklearn import metrics
from sklearn.model_selection import cross_validate, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import MultiLabelBinarizer, StandardScaler, OneHotEncoder
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.feature_selection import SelectKBest
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_squared_error, r2_score

In [ ]:
# Load cleaned data from CSV files.
mida = pd.read_csv("/content/mida.csv")
fow = pd.read_csv("/content/fow.csv") # Factors of war

# Restricting data to only after 1989, since that is when democratic index(the newest data sorce) starts
prepared_df = fow[fow['year'] >= 1989]

/tmp/ipykernel_22682/1055235076.py:3: DtypeWarning: Columns (4,7,8,11,15,19,22) have mixed types. Specify dtype option on import or set low_memory=False.
  fow = pd.read_csv("/content/fow.csv")


# Preparing and splitting data

In [ ]:
# Removing all na rows, since there are a low of empty rows
prepared_df = prepared_df.dropna()

# Selecting predictor variables
X = prepared_df[['ccode', 'year', 'cinc',  'sb_exist_cy', 'sb_dyad_count_cy', 'gdp', 'gdp_percent_change',
               'sb_intrastate_exist_cy', 'sb_intrastate_dyad_count_cy', 'sb_intrastate_main_govt_inv_incomp_cy', 'sb_interstate_exist_cy', 'sb_interstate_dyad_count_cy',
        'sb_interstate_main_govt_inv_incomp_cy', 'ns_exist_cy', 'ns_dyad_count_cy', 'os_exist_cy', 'os_dyad_count_cy',
        'os_main_govt_inv_cy', 'os_nsgroup_inv_cy','v2x_polyarchy', 'v2x_libdem', 'v2x_partipdem',
               'v2x_delibdem', 'v2x_egaldem', "region_cy", "sb_dyad_ids_cy", "ns_dyad_ids_cy",
               "sb_intrastate_dyad_ids_cy", "sb_interstate_dyad_ids_cy", "os_dyad_ids_cy", 'nat_res_rent'
               ]]

# Selecting target variables
y = prepared_df[['ctd_parties', 'ctd_civ', 'ctd_unk', 'ctd_best']]

# Splitting data for training and testing
X_train, X_test, y_train, y_test = sk.model_selection.train_test_split(X, y, test_size=0.2, shuffle=True, random_state=30)

In [ ]:
# Column names of columns with id arrays for values, they need to be preprocessed for the models
id_cols = [
    "sb_dyad_ids_cy",
    "ns_dyad_ids_cy",
    "sb_intrastate_dyad_ids_cy",
    "sb_interstate_dyad_ids_cy",
    "os_dyad_ids_cy"
]

# Column names of columsn with numerical values, they need to be scaled
numeric_col = ['cinc', 'sb_dyad_count_cy',
               'sb_intrastate_dyad_count_cy', 'sb_interstate_dyad_count_cy',
        'ns_dyad_count_cy',  'os_dyad_count_cy',
        'v2x_polyarchy', 'v2x_libdem', 'v2x_partipdem',
               'v2x_delibdem', 'v2x_egaldem', 'gdp', 'nat_res_rent'
  ]

# Convert id arrays which are stored as strings into actual python arrays
for col in id_cols:
    X_train[col] = X_train[col].apply(ast.literal_eval)
    X_test[col] = X_test[col].apply(ast.literal_eval)

In [ ]:
# Preprocess columns of ids data, arrays of ids data and numerical values
# Creates class that encodes variable length arrays into an array
class MultiLabelBinarizerWrapper(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.mlb = MultiLabelBinarizer()

    def fit(self, X, y=None):
        self.mlb.fit(X.squeeze())
        return self

    def transform(self, X):
        return self.mlb.transform(X.squeeze())

# Creates data preprocessor, scale numerical values, one hot encodes dingle id columns, encodes columns of arrays of ids
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_col),
        ("cat",
         OneHotEncoder(handle_unknown="ignore"),
         ["ccode","region_cy"]),

        ("sb_dyad",
         MultiLabelBinarizerWrapper(),
         ["sb_dyad_ids_cy"]),

        ("ns_dyad",
         MultiLabelBinarizerWrapper(),
         ["ns_dyad_ids_cy"]),

        ("intrastate",
         MultiLabelBinarizerWrapper(),
         ["sb_intrastate_dyad_ids_cy"]),

        ("interstate",
         MultiLabelBinarizerWrapper(),
         ["sb_interstate_dyad_ids_cy"]),

        ("os_dyad",
         MultiLabelBinarizerWrapper(),
         ["os_dyad_ids_cy"])
    ],
    remainder="passthrough"
)

# Fit and transform on training data
enc_X_train = preprocessor.fit_transform(X_train)
# Transform test data
enc_X_test = preprocessor.transform(X_test)

/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_label.py:909: UserWarning: unknown class(es) [14134, 14249, 575, 759, 762, 795, 796, 849, 866, 868] will be ignored
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_label.py:909: UserWarning: unknown class(es) [10247, 10301, 11241, 11942, 11947, 12004, 12090, 12284, 12342, 13792, 13793, 13796, 13798, 13812, 13899, 14053, 14152, 14251, 14341, 14342, 14344, 14347, 14348, 14372, 14374, 14393, 14750, 15018, 15019, 15023, 15028, 15051, 15564, 15565, 15566, 15675, 16006, 16021, 16031, 16071, 16220, 16223, 16248, 16259, 16260, 16277, 16294, 16295, 16300, 16302, 16309, 16312, 16313, 16315, 16319, 16325, 16327, 16331, 16436, 16749, 5186, 5199, 5228, 5237, 5239, 5250, 5259, 5260, 5274, 5291, 5327, 5328, 5357, 5365, 5393, 5399, 5401, 5412, 5413, 5414, 5478, 5490, 5492, 5493, 5496, 5498, 5499, 5506, 5561, 5562, 5566, 5575, 5596] will be ignored
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/skle

In [ ]:
# Helper function that optimizes hyper parameters throguh grid search over the target variables and returns best models and its test scores for each target variable
def get_best_models(grid_search, pred_targets, X_train, Y_train, X_test, Y_test):
  best_models = []
  best_models_scores = []

  # Loop throgh target variables
  for predict in pred_targets:
    #Find best hyper parameter for model
    print(predict)
    grid_search.fit(X_train, Y_train[predict])

    # Select best model and gets predictions on test values
    print(grid_search.best_params_)
    best_model = grid_search.best_estimator_
    y_pred = grid_search.predict(X_test)

    # Save root mean squared error and r2 score for model on test data
    root_mean = sk.metrics.root_mean_squared_error(Y_test[predict], y_pred)
    r2_score = sk.metrics.r2_score(Y_test[predict], y_pred)
    print(f"Root Mean: {root_mean}, r2: {r2_score}")

    best_models.append(clone(grid_search))
    best_models_scores.append((root_mean, r2_score))

  return best_models, best_models_scores

# Linear Model

In [ ]:
# Defines ridge model
ridge_model = Ridge(random_state=0)

# Define hyperparameters for grid search
cv_params = {'alpha': [0.001,0.01,0.1, 1, 10, 100, 250, 500, 1000]}

# Deine the scoring metrics
scoring = ['neg_root_mean_squared_error', 'r2']

# Instantiate grid search cross validation
ridge_cv = GridSearchCV(ridge_model, cv_params, scoring=scoring, refit="neg_root_mean_squared_error", cv=5,n_jobs=4)

In [ ]:
# Get best ridge model and prints its metrics for each target variable
linear_models, linear_models_scores = get_best_models(ridge_cv, y_train.columns, enc_X_train, y_train, enc_X_test, y_test)

ctd_parties
{'alpha': 10}
Root Mean: 1716.6373625237861, r2: 0.6027160446057505
ctd_civ
{'alpha': 1000}
Root Mean: 1758.5992893817188, r2: -1.7544764232781587
ctd_unk
{'alpha': 100}
Root Mean: 415.56404894907394, r2: -0.5903794214227456
ctd_best
{'alpha': 1000}
Root Mean: 3393.3297028369475, r2: 0.2194782529232333


# KNN model

In [ ]:
# Create a pipeline for knn so that we can range over different number of features selected
knn_pipeline = Pipeline([
    ('selector', SelectKBest()),
    ('knn', KNeighborsRegressor())
])

# Define hyperparameters for CV to search over
cv_params = {'knn__n_neighbors': [3,4,6,8,10,12],
             'knn__weights': ['uniform', 'distance'],
             'selector__k': [2, 5, 10, 15, 20, 25, 'all'],
             }

# Deine the scoring metrics
scoring = ['neg_root_mean_squared_error', 'r2']

# Instantiate grid search cross validation
knn_cv = GridSearchCV(knn_pipeline, cv_params, scoring=scoring, refit="neg_root_mean_squared_error", cv=5,n_jobs=4, error_score='raise')

In [ ]:
from sklearn.feature_selection import VarianceThreshold

# Recommended approach: remove zero-variance features first
selector = VarianceThreshold()
reduced_X_train = selector.fit_transform(enc_X_train)

In [ ]:
# Get best knn model and prints its metrics for each target variable
knn_models, knn_models_scores = get_best_models(knn_cv, y_train.columns, enc_X_train, y_train, enc_X_test, y_test)

ctd_parties


/usr/local/lib/python3.12/dist-packages/sklearn/feature_selection/_univariate_selection.py:112: RuntimeWarning: divide by zero encountered in divide
  f = msb / msw


{'knn__n_neighbors': 3, 'knn__weights': 'distance', 'selector__k': 'all'}
Root Mean: 1844.4644194502946, r2: 0.5413467570817789
ctd_civ


/usr/local/lib/python3.12/dist-packages/sklearn/feature_selection/_univariate_selection.py:112: RuntimeWarning: divide by zero encountered in divide
  f = msb / msw


{'knn__n_neighbors': 3, 'knn__weights': 'distance', 'selector__k': 25}
Root Mean: 503.0650564371593, r2: 0.774600174149256
ctd_unk


/usr/local/lib/python3.12/dist-packages/sklearn/feature_selection/_univariate_selection.py:112: RuntimeWarning: divide by zero encountered in divide
  f = msb / msw


{'knn__n_neighbors': 3, 'knn__weights': 'distance', 'selector__k': 'all'}
Root Mean: 272.07101600098196, r2: 0.31830616520134114
ctd_best
{'knn__n_neighbors': 10, 'knn__weights': 'distance', 'selector__k': 15}
Root Mean: 3874.090986921491, r2: -0.017354998942957556


/usr/local/lib/python3.12/dist-packages/sklearn/feature_selection/_univariate_selection.py:112: RuntimeWarning: divide by zero encountered in divide
  f = msb / msw


# Using random forest for prediction


In [ ]:
# Define random forest model
rf_model = RandomForestRegressor(random_state=0)

# Define hyperparameters for grid search
cv_params = {'max_depth': [3,4,6,8,10,12],
             'min_samples_leaf': [1,2],
             'min_samples_split': [2,3,4,5],
             'max_features': [5, 10, 20, 25, 31],
             'n_estimators': [75, 100, 125, 150, 175]
             }

# Deine the scoring metrics
scoring = ['neg_root_mean_squared_error', 'r2']

# Instantiate the grid search cross validation
rf_cv = GridSearchCV(rf_model, cv_params, scoring=scoring, refit="neg_root_mean_squared_error", cv=5,n_jobs=4, error_score='raise')

In [ ]:
# Get best rf model and prints its metrics for each target variable
rf_models = get_best_models(rf_cv, y_train.columns, enc_X_train, y_train, enc_X_test, y_test)


ctd_parties
{'max_depth': 8, 'max_features': 31, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 75}
Root Mean: 1982.7501181637278, r2: 0.4699951136257896
ctd_civ
{'max_depth': 4, 'max_features': 10, 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 125}
Root Mean: 1018.8437675960278, r2: 0.07547116879714777
ctd_unk
{'max_depth': 8, 'max_features': 31, 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 125}
Root Mean: 327.4183795674306, r2: 0.012741414814767205
ctd_best
{'max_depth': 4, 'max_features': 10, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 125}
Root Mean: 3242.082842713886, r2: 0.2875061722107879


# Using Gradient Boosting for prediction

In [ ]:
# Define gradiant boosting model
gb_model = GradientBoostingRegressor(random_state=0)

# Define hyperparameters for grid search
cv_params = {'learning_rate': [0.01, 0.05, 0.1, 0.15, 0.2, 0.25, 0.5, 1],
             'min_samples_leaf': [1,2,3],
             'min_samples_split': [2,3,4],
             'max_features': [5, 10, 20, 25, 31],
             'n_estimators': [75, 100, 125, 150, 175]
             }

# Deine the scoring metrics
scoring = ['neg_root_mean_squared_error', 'r2']

# Instantiate grid search cross validation
gb_cv = GridSearchCV(gb_model, cv_params, scoring=scoring, refit="neg_root_mean_squared_error", cv=5,n_jobs=6)

In [ ]:
# Get best gb model and prints its metrics for each target variable
gb_models = get_best_models(gb_cv, y_train.columns, enc_X_train, y_train, enc_X_test, y_test)

ctd_parties
{'learning_rate': 0.25, 'max_features': 2, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 125}
Root Mean: 1895.502509538024, r2: 0.5156128291246708
ctd_civ
{'learning_rate': 0.05, 'max_features': 2, 'min_samples_leaf': 1, 'min_samples_split': 3, 'n_estimators': 125}
Root Mean: 2658.791019467019, r2: -5.296129892234579
ctd_unk
{'learning_rate': 0.25, 'max_features': 5, 'min_samples_leaf': 3, 'min_samples_split': 2, 'n_estimators': 125}
Root Mean: 485.43279684227304, r2: -1.1701164503069803
ctd_best
{'learning_rate': 0.1, 'max_features': 2, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 125}
Root Mean: 5727.0989498733015, r2: -1.2233214814448452


# Neural Network

In [ ]:
# Splitting existing training set into training and evaluation set
nn_enc_X_train, X_val, nn_y_train, y_val = sk.model_selection.train_test_split(enc_X_train, y_train, test_size=0.2, shuffle=True, random_state=30)

In [ ]:
# Transforming all data into tensors for neural networks data loaders
tens_X_train = torch.tensor(nn_enc_X_train, dtype=torch.float32)
tens_X_test = torch.tensor(enc_X_test, dtype=torch.float32)
tens_X_val = torch.tensor(X_val, dtype=torch.float32)

tens_y_train = torch.tensor(nn_y_train.values, dtype=torch.float32)
tens_y_test = torch.tensor(y_test.values, dtype=torch.float32)
tens_y_val = torch.tensor(y_val.values, dtype=torch.float32)

In [ ]:
# Function to create a dataloader basedd on batch size, used later for hyper tuning batch size
def make_loader(X, y, batch_size):
    dataset = TensorDataset(X,y)

    return DataLoader(dataset, batch_size=batch_size, shuffle=True)

In [ ]:
# Neural netwrok architecture for predicting fow target variables
class Network(nn.Module):
    # Uses all 31 inputs to predict 4 target variables
    def __init__(self, input_dim=31, hidden1=128, hidden2=64, dropout=0.2):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(input_dim,hidden1),
            nn.ReLU(),

            # Normalized hiden value output to reduce over and under fitting
            nn.BatchNorm1d(hidden1),

            nn.Linear(hidden1,hidden2),
            nn.ReLU(),

            nn.BatchNorm1d(hidden2),

            # Regularizes model and reduces overfitting
            nn.Dropout(dropout),

            nn.Linear(hidden2,4)
        )

    def forward(self,x):
        return self.net(x)

#
model = Network()

In [ ]:
# Trains model and gets evaluation score, used later for hyper paramter tuning
def train_model(model, train_loader, val_loader, lr=1e-3, epochs=200):
    # Similar to mean square error, but better for noisy data
    criterion = nn.SmoothL1Loss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    best_val_loss = float("inf")
    best_state = None

    for epoch in range(epochs):
        # Trains the model over train loader
        model.train()

        train_loss = 0

        for xb,yb in train_loader:
            preds = model(xb)
            loss = criterion(preds, yb)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_loss += loss.item()

        # Validate model over data
        model.eval()
        val_loss = 0

        with torch.no_grad():
            for xb,yb in val_loader:
                preds = model(xb)

                loss = criterion(preds, yb)
                val_loss += loss.item()

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = (model.state_dict())

        if epoch == 199:
            print(f"Epoch {epoch}, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")

    model.load_state_dict(best_state)

    return model, best_val_loss

In [ ]:
# Evaluation and hyper parameter tuning
# Parameters for parameter grid to search over
param_grid = {
    "hidden1":[32,64,128,256],
    "hidden2":[32,64,128,256],
    "dropout":[0.0,.2,.4, 0.6],
    "lr":[0.0005, 0.001],
    "batch_size":[32,64]
}

best_model = None
best_score = float("inf")
best_params = None

# Loops through all different parameter combinations
for params in ParameterGrid(param_grid):
    print("\nTrying:", params)

    # Makes train and eval loader with varying batch sizes based on loop
    train_loader = make_loader(tens_X_train, tens_y_train, params["batch_size"])
    val_loader = make_loader(tens_X_val, tens_y_val, params["batch_size"])

    # Defin model with varying parameters, then trains it and gets evaluation score
    model = Network(input_dim=tens_X_train.shape[1], hidden1=params["hidden1"], hidden2=params["hidden2"], dropout=params["dropout"])
    model, val_loss = train_model(model, train_loader, val_loader,lr=params["lr"], epochs=200)

    # Saves model if val score is better than the best_model
    if val_loss < best_score:
        best_score = val_loss
        best_model = model
        best_params = params

print(f"\nBest Parameters: {best_params}")
print(f"Best Validation Loss: {best_score}")



Trying: {'batch_size': 32, 'dropout': 0.0, 'hidden1': 64, 'hidden2': 32, 'lr': 0.001}
Epoch 1, Train Loss: 29713.5793, Val Loss: 3887.1651

Trying: {'batch_size': 32, 'dropout': 0.0, 'hidden1': 64, 'hidden2': 32, 'lr': 0.0005}
Epoch 1, Train Loss: 29036.6263, Val Loss: 3748.9735

Trying: {'batch_size': 32, 'dropout': 0.0, 'hidden1': 64, 'hidden2': 64, 'lr': 0.001}
Epoch 1, Train Loss: 28951.0492, Val Loss: 4056.4562

Trying: {'batch_size': 32, 'dropout': 0.0, 'hidden1': 64, 'hidden2': 64, 'lr': 0.0005}
Epoch 1, Train Loss: 29037.8171, Val Loss: 4093.9014

Trying: {'batch_size': 32, 'dropout': 0.0, 'hidden1': 64, 'hidden2': 128, 'lr': 0.001}
Epoch 1, Train Loss: 28898.4174, Val Loss: 3948.3561

Trying: {'batch_size': 32, 'dropout': 0.0, 'hidden1': 64, 'hidden2': 128, 'lr': 0.0005}
Epoch 1, Train Loss: 29025.0452, Val Loss: 4198.0145

Trying: {'batch_size': 32, 'dropout': 0.0, 'hidden1': 128, 'hidden2': 32, 'lr': 0.001}
Epoch 1, Train Loss: 29005.7270, Val Loss: 3845.9384

Trying: {'bat

In [ ]:
# Testing of model
best_model.eval()

# Get best models predictions on test set
with torch.no_grad():
    preds = best_model(tens_X_test)

# Gets r2 and rmse metrics for each model and prints it
for i, col in enumerate(y_train.columns):
    network_r2_score = r2_score(tens_y_test[:,i], preds[:,i])
    network_rmse_score = np.sqrt(mean_squared_error(tens_y_test[:,i], preds[:,i]))
    print(col)
    print(f"r2: {network_r2_score}")
    print(f"rmse": {network_rmse_score})

In [ ]:
#Save best neural network model
torch.save(best_model, 'model.pth')

In [ ]:
# Saving knn models
with open('knn_models.pkl', 'wb') as f:
    pickle.dump(knn_models, f)
# Saving randomforest
with open('rf_models.pkl', 'wb') as f:
    pickle.dump(rf_models, f)
# Saving gradiantboosting
with open('gb_models.pkl', 'wb') as f:
    pickle.dump(gb_models, f)

In [ ]:
# Loading knn
with open('knn_models.pkl', 'rb') as f:
    knn_models = pickle.load(f)
# Loading ranodom forest
with open('rf_models.pkl', 'rb') as f:
    rf_models = pickle.load(f)
# Loading gradiant boosting
with open('gb_models.pkl', 'rb') as f:
    gb_models = pickle.load(f)
# Loading network
best_model = torch.load('model.pth', weights_only=False)

In [ ]:
mida[mida['fatalpre']!=0]

,dispnum,styear,endyear,fatality,fatalpre,hostlev,recip,numa,numb,ongo2014,meandur,acountries,bcountries
2,4,1946,1946,2,-9,4,1,1,1,0,183.0,[339],[200]
3,7,1951,1952,2,-9,4,1,1,1,0,106.0,[200],[651]
4,8,1856,1857,6,-9,5,1,1,1,0,242.0,[200],[630]
8,13,1863,1863,-9,-9,4,0,3,1,0,134.0,[220 200 2],[740]
12,19,1848,1849,6,-9,5,1,1,3,0,434.0,[300],[332 325 337]
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2403,4693,2013,2014,1,2,4,1,1,1,1,329.0,[702],[703]
2417,4708,2011,2011,1,1,4,1,1,1,0,161.0,[704],[702]
2423,4714,2014,2014,1,1,4,1,1,1,0,6.0,[771],[775]
2424,4715,2011,2011,-9,-9,4,0,1,1,0,2.0,[775],[800]
